In [0]:
parameters = []

In [0]:
table_list = []

for table_n in spark.catalog.listTables("walmart.silver"):
    table_list.append(table_n.name)

In [0]:
keep_columns = {
    "orders": [
        "order_id",
        "store_id",
        "order_timestamp",
        "payment_method",
        "order_status",
        "total_amount"
    ],
    "customers": [
        "customer_id"
    ],
    "order_items": [
        "order_item_id",
        "quantity",
        "unit_price",
        "line_amount"
    ],
    "products": [
        "product_id",
        "product_name",
        "category",
        "brand",
        "price"
    ],
    "employees": [
        "employee_id",
        "job_title",
        "salary"
    ],
    "stores": [
        "store_name"
    ]
}

In [0]:
for table_name in table_list:
    if not table_name.startswith("obt"):

        obt_dic = {}

        df = spark.table(f"walmart.silver.{table_name}")

        obt_dic["table"] = f"walmart.silver.{table_name}"
        obt_dic["alias"] = table_name

        columns = []

        for col_n in df.columns:

            if col_n in keep_columns.get(table_name, []):
                columns.append(f"{table_name}.{col_n}")

            else:
                columns.append(
                    f"{table_name}.{col_n} AS {table_name}_{col_n}"
                )

        if table_name == "orders":
            columns.append("current_timestamp() AS obt_processed_timestamp")

        obt_dic["col"] = ",".join(columns)

        if table_name == "customers":
            obt_dic["join_condition"] = \
                "orders.customer_id = customers.customer_id"

        elif table_name == "order_items":
            obt_dic["join_condition"] = \
                "orders.order_id = order_items.order_id"

        elif table_name == "products":
            obt_dic["join_condition"] = \
                "order_items.product_id = products.product_id"

        elif table_name == "employees":
            obt_dic["join_condition"] = \
                "orders.store_id = employees.store_id"

        elif table_name == "stores":
            obt_dic["join_condition"] = \
                "orders.store_id = stores.store_id"

        parameters.append(obt_dic)

In [0]:
from jinja2 import Template

In [0]:
jinja_obt_query = """
                    SELECT 
                        {% for param in parameters %}
                            {{ param['col'] }}{% if not loop.last %},{% endif %}
                        {% endfor %}
                    FROM 
                        {% for param in parameters %}
                            {% if param['table'] == 'walmart.silver.orders' %}
                                {{ param['table'] }} AS {{ param['alias'] }}
                            {% endif %}
                        {% endfor %}
                        {% for param in parameters %}
                            {% if param['table'] != 'walmart.silver.orders' %}
                                LEFT JOIN {{ param['table'] }} AS {{ param['alias'] }} 
                                ON {{ param['join_condition'] }}
                            {% endif %}
                        {% endfor %}
                  """

In [0]:
jinja_obt_query_text = Template(jinja_obt_query)
obt_query = jinja_obt_query_text.render(parameters=parameters)
